In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from tqdm import tqdm

In [ ]:
PROJECT_PATH = "/content/drive/MyDrive/Deepfake_Project"

FEATURE_PATH = os.path.join(PROJECT_PATH, "fusion_features")

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [ ]:
features = []
labels = []

# Original = 0
original_dir = os.path.join(FEATURE_PATH, "original")

for file in sorted(os.listdir(original_dir)):
    feature = np.load(os.path.join(original_dir, file))
    features.append(feature)
    labels.append(0)

# Deepfake = 1
deepfake_dir = os.path.join(FEATURE_PATH, "Deepfakes")

for file in sorted(os.listdir(deepfake_dir)):
    feature = np.load(os.path.join(deepfake_dir, file))
    features.append(feature)
    labels.append(1)

features = np.array(features)
labels = np.array(labels)

print(features.shape)
print(labels.shape)

In [ ]:
X_train, X_temp, y_train, y_temp = train_test_split(
    features,
    labels,
    test_size=0.2,
    random_state=42,
    stratify=labels
)

X_val, X_test, y_val, y_test = train_test_split(
    X_temp,
    y_temp,
    test_size=0.5,
    random_state=42,
    stratify=y_temp
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

In [ ]:
class DeepfakeDataset(Dataset):

    def __init__(self, features, labels):

        self.features = torch.tensor(
            features,
            dtype=torch.float32
        )

        self.labels = torch.tensor(
            labels,
            dtype=torch.long
        )

    def __len__(self):
        return len(self.features)

    def __getitem__(self, idx):

        return self.features[idx], self.labels[idx]

In [ ]:
train_dataset = DeepfakeDataset(X_train, y_train)
val_dataset = DeepfakeDataset(X_val, y_val)
test_dataset = DeepfakeDataset(X_test, y_test)

print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32,
    shuffle=False
)

print("DataLoaders Ready!")

In [ ]:
class DeepfakeClassifier(nn.Module):

    def __init__(self):
        super().__init__()

        self.network = nn.Sequential(

            nn.Linear(512, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Dropout(0.4),

            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),

            nn.Linear(256,128),
            nn.ReLU(),

            nn.Linear(128,2)

        )

    def forward(self,x):
        return self.network(x)

In [ ]:
model = DeepfakeClassifier().to(device)

criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=0.001,
    weight_decay=1e-4
)

In [ ]:
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
    optimizer,
    mode='max',
    factor=0.5,
    patience=3
)

In [ ]:
for features, labels in train_loader:

    print(features.shape)
    print(labels.shape)

    output = model(features.to(device))

    print(output.shape)

    break

In [ ]:
def train_one_epoch(model, loader, criterion, optimizer):

    model.train()

    running_loss = 0
    correct = 0
    total = 0

    for features, labels in tqdm(loader):

        features = features.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(features)

        loss = criterion(outputs, labels)

        loss.backward()

        optimizer.step()

        running_loss += loss.item()

        _, predicted = torch.max(outputs, 1)

        total += labels.size(0)

        correct += (predicted == labels).sum().item()

    epoch_loss = running_loss / len(loader)
    epoch_acc = 100 * correct / total

    return epoch_loss, epoch_acc

In [ ]:
def evaluate(model, loader):

    model.eval()

    predictions = []
    targets = []

    with torch.no_grad():

        for features, labels in loader:

            features = features.to(device)

            outputs = model(features)

            _, predicted = torch.max(outputs, 1)

            predictions.extend(predicted.cpu().numpy())

            targets.extend(labels.numpy())

    accuracy = accuracy_score(targets, predictions)
    precision = precision_score(targets, predictions)
    recall = recall_score(targets, predictions)
    f1 = f1_score(targets, predictions)

    return accuracy, precision, recall, f1

In [ ]:
# ===========================
# TRAINING WITH HISTORY
# ===========================

EPOCHS = 20

best_val_acc = 0
patience = 5
counter = 0

MODEL_PATH = "/content/drive/MyDrive/Deepfake_Project/best_model_label_smoothing.pth"

# History Lists
train_loss_history = []
train_acc_history = []

val_acc_history = []
precision_history = []
recall_history = []
f1_history = []

for epoch in range(EPOCHS):

    print(f"\nEpoch {epoch+1}/{EPOCHS}")

    # -------------------------
    # Train
    # -------------------------
    train_loss, train_acc = train_one_epoch(
        model,
        train_loader,
        criterion,
        optimizer
    )

    # -------------------------
    # Validation
    # -------------------------
    val_acc, precision, recall, f1 = evaluate(
        model,
        val_loader
    )

    scheduler.step(val_acc)

    print(f"Train Loss      : {train_loss:.4f}")
    print(f"Train Accuracy  : {train_acc:.2f}%")
    print(f"Validation Acc. : {val_acc:.4f}")
    print(f"Precision       : {precision:.4f}")
    print(f"Recall          : {recall:.4f}")
    print(f"F1 Score        : {f1:.4f}")

    # -------------------------
    # Store History
    # -------------------------
    train_loss_history.append(train_loss)
    train_acc_history.append(train_acc)

    val_acc_history.append(val_acc)
    precision_history.append(precision)
    recall_history.append(recall)
    f1_history.append(f1)

    # -------------------------
    # Save Best Model
    # -------------------------
    if val_acc > best_val_acc:

        best_val_acc = val_acc
        counter = 0

        torch.save(model.state_dict(), MODEL_PATH)

        print("✅ Best Model Saved!")

    else:

        counter += 1

    # -------------------------
    # Early Stopping
    # -------------------------
    if counter >= patience:

        print(f"\n🛑 Early stopping at Epoch {epoch+1}")
        break

print("\n====================================")
print("Training Completed")
print(f"Best Validation Accuracy : {best_val_acc:.4f}")
print(f"Model Saved At : {MODEL_PATH}")
print("====================================")

In [ ]:
import matplotlib.pyplot as plt

# Epoch numbers
epochs = range(1, len(train_loss_history) + 1)

# Convert validation accuracy to percentage
val_acc_plot = [x * 100 for x in val_acc_history]

# ==============================
# Figure 1: Training Loss
# ==============================
plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    train_loss_history,
    marker='o',
    linewidth=2,
    label="Training Loss"
)

plt.title("Training Loss vs Epoch", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Loss", fontsize=12)

plt.xticks(list(epochs))
plt.grid(True)
plt.legend()

plt.savefig(
    "/content/drive/MyDrive/Deepfake_Project/training_loss.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ==============================
# Figure 2: Training vs Validation Accuracy
# ==============================
plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    train_acc_history,
    marker='o',
    linewidth=2,
    label="Training Accuracy"
)

plt.plot(
    epochs,
    val_acc_plot,
    marker='s',
    linewidth=2,
    label="Validation Accuracy"
)

plt.title("Training and Validation Accuracy", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("Accuracy (%)", fontsize=12)

plt.xticks(list(epochs))
plt.ylim(70, 100)

plt.grid(True)
plt.legend()

plt.savefig(
    "/content/drive/MyDrive/Deepfake_Project/accuracy_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

# ==============================
# Figure 3: Validation F1 Score
# ==============================
plt.figure(figsize=(8,5))

plt.plot(
    epochs,
    f1_history,
    marker='o',
    linewidth=2,
    label="Validation F1 Score"
)

plt.title("Validation F1 Score", fontsize=14)
plt.xlabel("Epoch", fontsize=12)
plt.ylabel("F1 Score", fontsize=12)

plt.xticks(list(epochs))
plt.ylim(0.80, 0.86)

plt.grid(True)
plt.legend()

plt.savefig(
    "/content/drive/MyDrive/Deepfake_Project/f1_curve.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("====================================")
print("✅ All graphs generated successfully")
print("====================================")
print("Saved Files:")
print("1. training_loss.png")
print("2. accuracy_curve.png")
print("3. f1_curve.png")

In [ ]:
MODEL_PATH = "/content/drive/MyDrive/Deepfake_Project/best_model_label_smoothing.pth"

model.load_state_dict(torch.load(MODEL_PATH, map_location=device))
model.eval()

In [ ]:
targets, predictions, probabilities = test_model(
    model,
    test_loader
)

from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

accuracy = accuracy_score(targets, predictions)
precision = precision_score(targets, predictions)
recall = recall_score(targets, predictions)
f1 = f1_score(targets, predictions)

print("="*40)
print("TEST RESULTS")
print("="*40)

print(f"Accuracy : {accuracy:.4f}")
print(f"Precision: {precision:.4f}")
print(f"Recall   : {recall:.4f}")
print(f"F1 Score : {f1:.4f}")

In [ ]:
print(classification_report(
    targets,
    predictions,
    target_names=["Original", "Deepfake"]
))

In [ ]:
disp.plot(cmap="Blues")
plt.title("Confusion Matrix")
plt.savefig("confusion_matrix.png", dpi=300, bbox_inches="tight")
plt.show()

In [ ]:
from sklearn.metrics import roc_curve, auc

fpr, tpr, thresholds = roc_curve(targets, probabilities)
roc_auc = auc(fpr, tpr)

plt.figure(figsize=(6,6))
plt.plot(fpr, tpr, linewidth=2, label=f"AUC = {roc_auc:.4f}")
plt.plot([0,1],[0,1],'k--')

plt.xlabel("False Positive Rate")
plt.ylabel("True Positive Rate")
plt.title("ROC Curve")
plt.legend()

plt.savefig("roc_curve.png", dpi=300, bbox_inches="tight")
plt.show()

print("AUC :", roc_auc)

In [ ]:
from sklearn.metrics import classification_report

report = classification_report(
    targets,
    predictions,
    target_names=["Original", "Deepfake"]
)

print(report)

with open("classification_report.txt", "w") as f:
    f.write(report)

print("✅ Classification Report Saved")

In [ ]:
import shutil

shutil.copy(
    "classification_report.txt",
    "/content/drive/MyDrive/Deepfake_Project/classification_report.txt"
)

shutil.copy(
    "confusion_matrix.png",
    "/content/drive/MyDrive/Deepfake_Project/confusion_matrix.png"
)

shutil.copy(
    "roc_curve.png",
    "/content/drive/MyDrive/Deepfake_Project/roc_curve.png"
)

print("✅ Files copied to Google Drive")

In [ ]:
import matplotlib.pyplot as plt

metrics = ["Accuracy", "Precision", "Recall", "F1 Score", "AUC"]
values = [79.0, 78.43, 80.0, 79.21, 85.60]

plt.figure(figsize=(8,5))

bars = plt.bar(metrics, values)

plt.ylim(70, 90)

plt.ylabel("Percentage (%)")
plt.title("Evaluation Metrics")

# Values on top of bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.3,
        f"{height:.2f}",
        ha='center',
        fontsize=10
    )

plt.grid(axis='y', linestyle='--', alpha=0.5)

plt.savefig(
    "/content/drive/MyDrive/Deepfake_Project/evaluation_metrics.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("✅ Evaluation Metrics graph saved.")

In [ ]:
import matplotlib.pyplot as plt

datasets = ["Training", "Validation", "Test"]
accuracies = [98.88, 84.00, 79.00]

plt.figure(figsize=(7,5))

bars = plt.bar(
    datasets,
    accuracies,
    width=0.5
)

plt.ylim(70, 100)

plt.title("Accuracy Comparison Across Training, Validation, and Test Sets",
          fontsize=14)

plt.xlabel("Dataset", fontsize=12)
plt.ylabel("Accuracy (%)", fontsize=12)

# Display values on bars
for bar in bars:
    height = bar.get_height()
    plt.text(
        bar.get_x() + bar.get_width()/2,
        height + 0.5,
        f"{height:.2f}%",
        ha="center",
        fontsize=11,
        fontweight="bold"
    )

plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.savefig(
    "/content/drive/MyDrive/Deepfake_Project/train_val_test_accuracy.png",
    dpi=300,
    bbox_inches="tight"
)

plt.show()

print("✅ Train-Validation-Test Accuracy graph saved successfully!")